<a href="https://colab.research.google.com/github/Sasidhar015/Data-engineering-training-2026/blob/main/27-10-2026_PySpark_Exersices/PySpark_Hands_on_Exercise-2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**PySpark Hands-on Exercise - 2**

In [3]:
!pip -q install pyspark

In [1]:
from google.colab import files
uploaded = files.upload()

Saving pyspark_logistics_practice_set_2.zip to pyspark_logistics_practice_set_2 (1).zip


In [2]:
import os
import zipfile

zip_files = [name for name in uploaded.keys() if name.lower().endswith(".zip")]
if not zip_files:
    raise ValueError("Please upload the dataset ZIP file.")
zip_path = zip_files[0]
data_dir = "/content/logistics_data"
os.makedirs(data_dir, exist_ok=True)

with zipfile.ZipFile(zip_path, "r") as zf:
    zf.extractall(data_dir)

print("Extracted files:")
for name in sorted(os.listdir(data_dir)):
    print(name)

Extracted files:
carrier_master.csv
route_master.csv
shipments_75000.csv
shipments_extra_12000.csv
warehouse_master.csv
warehouse_sensor_config.json


In [4]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark import StorageLevel
import time
import os

spark = (SparkSession.builder
    .appName("LogisticsDeliveryPracticeSet02")
    .getOrCreate())
spark.sparkContext.setLogLevel("WARN")
print("Spark version:", spark.version)

Spark version: 4.0.4


In [5]:
shipments_path = f"{data_dir}/shipments_75000.csv"
extra_path = f"{data_dir}/shipments_extra_12000.csv"
warehouse_path = f"{data_dir}/warehouse_master.csv"
carrier_path = f"{data_dir}/carrier_master.csv"
route_path = f"{data_dir}/route_master.csv"
sensor_path = f"{data_dir}/warehouse_sensor_config.json"

shipments_df = (spark.read.option("header", True).option("inferSchema", True).csv(shipments_path))
extra_df = (spark.read.option("header", True).option("inferSchema", True).csv(extra_path))
warehouse_df = (spark.read.option("header", True).option("inferSchema", True).csv(warehouse_path))
carrier_df = ( spark.read.option("header", True).option("inferSchema", True).csv(carrier_path))
route_df = (spark.read.option("header", True).option("inferSchema", True).csv(route_path))
sensor_df = spark.read.option("multiline", True).json(sensor_path)

print("shipments_df:", shipments_df.count())
print("extra_df:", extra_df.count())
print("warehouse_df:", warehouse_df.count())
print("carrier_df:", carrier_df.count())
print("route_df:", route_df.count())
print("sensor_df:", sensor_df.count())

shipments_df: 75000
extra_df: 12000
warehouse_df: 8
carrier_df: 7
route_df: 12
sensor_df: 8


In [6]:
date_columns = ["booking_date", "dispatch_date","promised_delivery_date", "actual_delivery_date"]
for c in date_columns:
    shipments_df = shipments_df.withColumn(c, F.to_date(F.col(c)))
    extra_df = extra_df.withColumn(c, F.to_date(F.col(c)))
numeric_types = {
    "package_weight_kg": "double",
    "shipping_fee_inr": "double",
    "shipping_cost_inr": "double",
    "customer_rating": "double",
    "handling_minutes": "double",
    "delivery_attempts": "int"}
for c, t in numeric_types.items():
    shipments_df = shipments_df.withColumn(c, F.col(c).cast(t))
    extra_df = extra_df.withColumn(c, F.col(c).cast(t))
shipments_df.printSchema()

root
 |-- shipment_id: string (nullable = true)
 |-- booking_date: date (nullable = true)
 |-- dispatch_date: date (nullable = true)
 |-- promised_delivery_date: date (nullable = true)
 |-- actual_delivery_date: date (nullable = true)
 |-- warehouse_id: string (nullable = true)
 |-- carrier_id: string (nullable = true)
 |-- route_id: string (nullable = true)
 |-- customer_segment: string (nullable = true)
 |-- service_level: string (nullable = true)
 |-- package_weight_kg: double (nullable = true)
 |-- shipping_fee_inr: double (nullable = true)
 |-- shipping_cost_inr: double (nullable = true)
 |-- shipment_status: string (nullable = true)
 |-- delivery_attempts: integer (nullable = true)
 |-- damage_flag: string (nullable = true)
 |-- customer_rating: double (nullable = true)
 |-- handling_minutes: double (nullable = true)
 |-- payment_mode: string (nullable = true)
 |-- priority_flag: string (nullable = true)



**PART A | DataFrame Operations**

In [8]:
shipments_df.printSchema()
print("Total shipment records:", shipments_df.count())
shipments_df.show(15, truncate=False)

root
 |-- shipment_id: string (nullable = true)
 |-- booking_date: date (nullable = true)
 |-- dispatch_date: date (nullable = true)
 |-- promised_delivery_date: date (nullable = true)
 |-- actual_delivery_date: date (nullable = true)
 |-- warehouse_id: string (nullable = true)
 |-- carrier_id: string (nullable = true)
 |-- route_id: string (nullable = true)
 |-- customer_segment: string (nullable = true)
 |-- service_level: string (nullable = true)
 |-- package_weight_kg: double (nullable = true)
 |-- shipping_fee_inr: double (nullable = true)
 |-- shipping_cost_inr: double (nullable = true)
 |-- shipment_status: string (nullable = true)
 |-- delivery_attempts: integer (nullable = true)
 |-- damage_flag: string (nullable = true)
 |-- customer_rating: double (nullable = true)
 |-- handling_minutes: double (nullable = true)
 |-- payment_mode: string (nullable = true)
 |-- priority_flag: string (nullable = true)

Total shipment records: 75000
+-----------+------------+-------------+-----

In [9]:
ex02_df = shipments_df.select("shipment_id", "warehouse_id", "service_level","shipment_status", "shipping_fee_inr")
ex02_df.show(10, truncate=False)

+-----------+------------+-------------+---------------+----------------+
|shipment_id|warehouse_id|service_level|shipment_status|shipping_fee_inr|
+-----------+------------+-------------+---------------+----------------+
|SHP0000001 |W08         |Express      |Delivered      |554.34          |
|SHP0000002 |W08         |Standard     |Delivered      |414.56          |
|SHP0000003 |W01         |Express      |Delivered      |724.64          |
|SHP0000004 |W03         |Standard     |Delivered      |287.86          |
|SHP0000005 |W03         |Economy      |Delivered      |251.68          |
|SHP0000006 |W01         |Express      |Delivered      |477.9           |
|SHP0000007 |W02         |Standard     |In Transit     |298.68          |
|SHP0000008 |W03         |Economy      |In Transit     |284.52          |
|SHP0000009 |W02         |Standard     |Delivered      |861.74          |
|SHP0000010 |W02         |Economy      |Delivered      |309.76          |
+-----------+------------+------------

In [10]:
ex03_df = shipments_df.filter((F.col("service_level") == "Express") &(F.col("package_weight_kg") > 5))
ex03_df.show(20, truncate=False)

+-----------+------------+-------------+----------------------+--------------------+------------+----------+--------+----------------+-------------+-----------------+----------------+-----------------+---------------+-----------------+-----------+---------------+----------------+----------------+-------------+
|shipment_id|booking_date|dispatch_date|promised_delivery_date|actual_delivery_date|warehouse_id|carrier_id|route_id|customer_segment|service_level|package_weight_kg|shipping_fee_inr|shipping_cost_inr|shipment_status|delivery_attempts|damage_flag|customer_rating|handling_minutes|payment_mode    |priority_flag|
+-----------+------------+-------------+----------------------+--------------------+------------+----------+--------+----------------+-------------+-----------------+----------------+-----------------+---------------+-----------------+-----------+---------------+----------------+----------------+-------------+
|SHP0000003 |2026-01-31  |2026-02-01   |2026-02-03            |2

In [11]:
ex04_df = shipments_df.orderBy(F.col("shipping_fee_inr").desc_nulls_last())
ex04_df.show(20, truncate=False)

+-----------+------------+-------------+----------------------+--------------------+------------+----------+--------+----------------+-------------+-----------------+----------------+-----------------+---------------+-----------------+-----------+---------------+----------------+----------------+-------------+
|shipment_id|booking_date|dispatch_date|promised_delivery_date|actual_delivery_date|warehouse_id|carrier_id|route_id|customer_segment|service_level|package_weight_kg|shipping_fee_inr|shipping_cost_inr|shipment_status|delivery_attempts|damage_flag|customer_rating|handling_minutes|payment_mode    |priority_flag|
+-----------+------------+-------------+----------------------+--------------------+------------+----------+--------+----------------+-------------+-----------------+----------------+-----------------+---------------+-----------------+-----------+---------------+----------------+----------------+-------------+
|SHP0044904 |2026-08-11  |2026-08-13   |2026-08-16            |N

**PART B | Aggregations and Joins**

In [18]:
ex05_df = shipments_df.groupBy("shipment_status").agg(F.count("*").alias("shipment_count")).orderBy(F.desc("shipment_count"))
ex05_df.show(truncate=False)

+---------------+--------------+
|shipment_status|shipment_count|
+---------------+--------------+
|Delivered      |52434         |
|In Transit     |10472         |
|Returned       |6130          |
|Cancelled      |3723          |
|Lost           |2241          |
+---------------+--------------+



In [19]:
ex06_df = shipments_df.groupBy("service_level").agg(
    F.sum("shipping_fee_inr").alias("total_shipping_fees_inr"),
    F.sum("shipping_cost_inr").alias("total_shipping_costs_inr"),
    F.sum(
        F.when(
            F.col("shipping_cost_inr").isNotNull(),
            F.col("shipping_fee_inr") - F.col("shipping_cost_inr"))).alias("profit_on_rows_with_known_cost_inr")).withColumn(
    "profit_margin_pct",
    F.when(
        F.col("total_shipping_fees_inr") != 0,
        F.col("profit_on_rows_with_known_cost_inr") /
        F.col("total_shipping_fees_inr") * 100))
ex06_df.show(truncate=False)

+-------------+-----------------------+------------------------+----------------------------------+------------------+
|service_level|total_shipping_fees_inr|total_shipping_costs_inr|profit_on_rows_with_known_cost_inr|profit_margin_pct |
+-------------+-----------------------+------------------------+----------------------------------+------------------+
|Express      |1.0058468059999986E7   |7350805.699999983       |2514772.519999993                 |25.001546010774895|
|Standard     |1.3437372619999977E7   |1.0110465900000025E7    |3043793.239999998                 |22.651699302210783|
|Economy      |4489918.680000002      |3386212.500000007       |1018119.2000000007                |22.67567126627782 |
+-------------+-----------------------+------------------------+----------------------------------+------------------+



In [20]:
ex07_df = (shipments_df.join(warehouse_df.select("warehouse_id", "region"), "warehouse_id", "left")
    .groupBy("region")
    .agg(F.count("*").alias("shipment_count"))
    .orderBy(F.desc("shipment_count")))
ex07_df.show(truncate=False)

+------+--------------+
|region|shipment_count|
+------+--------------+
|South |33018         |
|West  |26909         |
|North |9767          |
|East  |5306          |
+------+--------------+



In [21]:
ex08_df = (shipments_df
    .join(carrier_df.select("carrier_id", "carrier_name"), "carrier_id", "left")
    .join(route_df.select("route_id", "distance_km"), "route_id", "left")
    .filter(
        F.col("shipping_cost_inr").isNotNull() &
        F.col("distance_km").isNotNull() &
        (F.col("distance_km") > 0))
    .withColumn("cost_per_km", F.col("shipping_cost_inr") / F.col("distance_km"))
    .groupBy("carrier_id", "carrier_name")
    .agg(F.avg("cost_per_km").alias("avg_shipping_cost_per_km"))
    .orderBy("avg_shipping_cost_per_km"))
ex08_df.show(truncate=False)

+----------+--------------------+------------------------+
|carrier_id|carrier_name        |avg_shipping_cost_per_km|
+----------+--------------------+------------------------+
|C06       |RouteWise           |0.5879324060720517      |
|C05       |CargoNest           |0.5897283901696387      |
|C03       |MetroRoute Logistics|0.594275960971091       |
|C02       |ParcelBridge        |0.5957409571811839      |
|C01       |SwiftLine Express   |0.6032748764541479      |
|C07       |BluePeak Delivery   |0.6074832283099014      |
|C04       |AirSprint Couriers  |0.6125755103088726      |
|NULL      |NULL                |0.6318401277501529      |
+----------+--------------------+------------------------+



**PART C | Null Handling and Conditional Logic**

In [23]:
ex09_df = shipments_df.select(*[
    F.sum(F.col(c).isNull().cast("int")).alias(f"missing_{c}")
    for c in [
        "carrier_id", "package_weight_kg",
        "shipping_cost_inr", "customer_rating"]])
ex09_df.show(truncate=False)

+------------------+-------------------------+-------------------------+-----------------------+
|missing_carrier_id|missing_package_weight_kg|missing_shipping_cost_inr|missing_customer_rating|
+------------------+-------------------------+-------------------------+-----------------------+
|871               |1129                     |1503                     |26255                  |
+------------------+-------------------------+-------------------------+-----------------------+



In [25]:
handling_filled_df = shipments_df.na.fill({"handling_minutes": 0})
carrier_present_df = shipments_df.na.drop(subset=["carrier_id"])

print("Original records:", shipments_df.count())
print("Rows with handling_minutes filled:", handling_filled_df.count())
print("Rows after dropping missing carrier_id:", carrier_present_df.count())

handling_filled_df.select("shipment_id", "handling_minutes").show(10)

Original records: 75000
Rows with handling_minutes filled: 75000
Rows after dropping missing carrier_id: 74129
+-----------+----------------+
|shipment_id|handling_minutes|
+-----------+----------------+
| SHP0000001|           122.0|
| SHP0000002|           220.0|
| SHP0000003|           188.0|
| SHP0000004|           179.0|
| SHP0000005|           218.0|
| SHP0000006|            48.0|
| SHP0000007|           214.0|
| SHP0000008|            76.0|
| SHP0000009|           222.0|
| SHP0000010|            98.0|
+-----------+----------------+
only showing top 10 rows


In [27]:
ex11_df = shipments_df.withColumn(
    "weight_category",
    F.when(F.col("package_weight_kg").isNull(), "Unknown")
     .when(F.col("package_weight_kg") < 2, "Light")
     .when(F.col("package_weight_kg") <= 10, "Medium")
     .otherwise("Heavy"))
ex11_df.select("shipment_id", "package_weight_kg", "weight_category").show(20)

+-----------+-----------------+---------------+
|shipment_id|package_weight_kg|weight_category|
+-----------+-----------------+---------------+
| SHP0000001|             3.93|         Medium|
| SHP0000002|             5.62|         Medium|
| SHP0000003|             8.78|         Medium|
| SHP0000004|             0.97|          Light|
| SHP0000005|             NULL|        Unknown|
| SHP0000006|             6.05|         Medium|
| SHP0000007|             1.36|          Light|
| SHP0000008|             2.04|         Medium|
| SHP0000009|            18.73|          Heavy|
| SHP0000010|             2.02|         Medium|
| SHP0000011|             6.87|         Medium|
| SHP0000012|            11.83|          Heavy|
| SHP0000013|             3.73|         Medium|
| SHP0000014|             1.42|          Light|
| SHP0000015|             1.74|          Light|
| SHP0000016|             6.87|         Medium|
| SHP0000017|             2.58|         Medium|
| SHP0000018|             1.85|         

**PART D | Date and Time Functions**

In [29]:
ex12_df = (
    shipments_df
    .filter(
        (F.col("shipment_status") == "Delivered") &
        F.col("dispatch_date").isNotNull() &
        F.col("actual_delivery_date").isNotNull())
    .withColumn(
        "delivery_duration_days",
        F.datediff("actual_delivery_date", "dispatch_date")))
ex12_df.select(
    "shipment_id", "dispatch_date", "actual_delivery_date",
    "delivery_duration_days").show(20, truncate=False)

+-----------+-------------+--------------------+----------------------+
|shipment_id|dispatch_date|actual_delivery_date|delivery_duration_days|
+-----------+-------------+--------------------+----------------------+
|SHP0000001 |2026-05-24   |2026-05-27          |3                     |
|SHP0000002 |2026-05-18   |2026-05-21          |3                     |
|SHP0000003 |2026-02-01   |2026-02-04          |3                     |
|SHP0000004 |2026-03-12   |2026-03-18          |6                     |
|SHP0000005 |2026-08-16   |2026-08-29          |13                    |
|SHP0000006 |2026-02-08   |2026-02-09          |1                     |
|SHP0000009 |2026-08-12   |2026-08-21          |9                     |
|SHP0000010 |2026-06-02   |2026-06-11          |9                     |
|SHP0000011 |2026-04-27   |2026-05-01          |4                     |
|SHP0000013 |2026-06-26   |2026-06-29          |3                     |
|SHP0000014 |2026-04-07   |2026-04-09          |2               

In [31]:
ex13_df = shipments_df.withColumn(
    "delivery_performance",
    F.when(F.col("actual_delivery_date").isNull(), "Missing Actual Date")
     .when(F.col("actual_delivery_date") < F.col("promised_delivery_date"), "Early")
     .when(F.col("actual_delivery_date") == F.col("promised_delivery_date"), "On Time")
     .otherwise("Late"))
ex13_df.groupBy("delivery_performance").count().show()

+--------------------+-----+
|delivery_performance|count|
+--------------------+-----+
|             On Time|22049|
| Missing Actual Date|22566|
|                Late|21900|
|               Early| 8485|
+--------------------+-----+



In [33]:
ex14_df = (shipments_df
    .filter(F.col("shipment_status") == "Delivered")
    .withColumn("booking_month", F.date_format("booking_date", "yyyy-MM"))
    .groupBy("booking_month")
    .agg(
        F.count("*").alias("delivered_shipment_count"),
        F.sum("shipping_fee_inr").alias("total_shipping_fees_inr")).orderBy("booking_month"))
ex14_df.show(truncate=False)

+-------------+------------------------+-----------------------+
|booking_month|delivered_shipment_count|total_shipping_fees_inr|
+-------------+------------------------+-----------------------+
|2026-01      |6710                    |2525138.440000005      |
|2026-02      |6097                    |2268113.5000000037     |
|2026-03      |6810                    |2530456.2400000026     |
|2026-04      |6379                    |2389163.4600000014     |
|2026-05      |6858                    |2549683.9999999963     |
|2026-06      |6615                    |2447590.1200000006     |
|2026-07      |6773                    |2512355.2199999997     |
|2026-08      |6192                    |2306432.5199999996     |
+-------------+------------------------+-----------------------+



**PART E | Window Functions**

In [34]:
region_carrier_fees_df = (shipments_df
    .join(warehouse_df.select("warehouse_id", "region"), "warehouse_id", "left")
    .join(carrier_df.select("carrier_id", "carrier_name"), "carrier_id", "left")
    .groupBy("region", "carrier_id", "carrier_name")
    .agg(F.sum("shipping_fee_inr").alias("total_shipping_fees_inr")))
region_fee_window = Window.partitionBy("region").orderBy(F.desc("total_shipping_fees_inr"))
ex15_df = region_carrier_fees_df.withColumn(
    "fee_rank", F.dense_rank().over(region_fee_window))
ex15_df.orderBy("region", "fee_rank").show(truncate=False)

+------+----------+--------------------+-----------------------+--------+
|region|carrier_id|carrier_name        |total_shipping_fees_inr|fee_rank|
+------+----------+--------------------+-----------------------+--------+
|East  |C01       |SwiftLine Express   |393862.61999999994     |1       |
|East  |C02       |ParcelBridge        |373590.02              |2       |
|East  |C03       |MetroRoute Logistics|302260.92000000004     |3       |
|East  |C05       |CargoNest           |243322.91999999998     |4       |
|East  |C07       |BluePeak Delivery   |222025.85999999993     |5       |
|East  |C04       |AirSprint Couriers  |211583.64000000004     |6       |
|East  |C06       |RouteWise           |195810.3399999999      |7       |
|East  |NULL      |NULL                |24594.6                |8       |
|North |C02       |ParcelBridge        |731442.0999999994      |1       |
|North |C01       |SwiftLine Express   |653223.6200000005      |2       |
|North |C03       |MetroRoute Logistic

In [36]:
latest_window = Window.partitionBy("warehouse_id").orderBy(
    F.col("booking_date").desc_nulls_last(),
    F.col("shipment_id").desc())
ex16_df = (
    shipments_df
    .withColumn("row_num", F.row_number().over(latest_window))
    .filter(F.col("row_num") == 1)
    .drop("row_num"))
ex16_df.show(truncate=False)

+-----------+------------+-------------+----------------------+--------------------+------------+----------+--------+----------------+-------------+-----------------+----------------+-----------------+---------------+-----------------+-----------+---------------+----------------+----------------+-------------+
|shipment_id|booking_date|dispatch_date|promised_delivery_date|actual_delivery_date|warehouse_id|carrier_id|route_id|customer_segment|service_level|package_weight_kg|shipping_fee_inr|shipping_cost_inr|shipment_status|delivery_attempts|damage_flag|customer_rating|handling_minutes|payment_mode    |priority_flag|
+-----------+------------+-------------+----------------------+--------------------+------------+----------+--------+----------------+-------------+-----------------+----------------+-----------------+---------------+-----------------+-----------+---------------+----------------+----------------+-------------+
|SHP0072346 |2026-08-28  |NULL         |2026-09-03            |N

In [37]:
running_fee_window = (
    Window.partitionBy("warehouse_id")
    .orderBy("booking_date", "shipment_id")
    .rowsBetween(Window.unboundedPreceding, Window.currentRow))
ex17_df = shipments_df.withColumn(
    "running_shipping_fee_inr",
    F.sum("shipping_fee_inr").over(running_fee_window))
ex17_df.select(
    "warehouse_id", "booking_date", "shipment_id",
    "shipping_fee_inr", "running_shipping_fee_inr").orderBy("warehouse_id", "booking_date", "shipment_id").show(20, truncate=False)

+------------+------------+-----------+----------------+------------------------+
|warehouse_id|booking_date|shipment_id|shipping_fee_inr|running_shipping_fee_inr|
+------------+------------+-----------+----------------+------------------------+
|W01         |2026-01-01  |SHP0004062 |312.48          |312.48                  |
|W01         |2026-01-01  |SHP0006739 |227.02          |539.5                   |
|W01         |2026-01-01  |SHP0007553 |316.02          |855.52                  |
|W01         |2026-01-01  |SHP0008505 |572.68          |1428.1999999999998      |
|W01         |2026-01-01  |SHP0009674 |322.9           |1751.1                  |
|W01         |2026-01-01  |SHP0011312 |390.1           |2141.2                  |
|W01         |2026-01-01  |SHP0011588 |421.72          |2562.92                 |
|W01         |2026-01-01  |SHP0011698 |190.22          |2753.14                 |
|W01         |2026-01-01  |SHP0011973 |384.64          |3137.7799999999997      |
|W01         |20

**PART F | Union and Complex JSON**

In [38]:
combined_df = shipments_df.unionByName(extra_df)
print("Original shipments:", shipments_df.count())
print("Extra batch:", extra_df.count())
print("Combined records:", combined_df.count())

Original shipments: 75000
Extra batch: 12000
Combined records: 87000


In [39]:
duplicate_ids_df = (
    combined_df.groupBy("shipment_id")
    .count()
    .filter(F.col("count") > 1)
    .orderBy(F.desc("count"), "shipment_id"))
print("Repeated shipment IDs:")
duplicate_ids_df.show(50, truncate=False)

before_dedup_count = combined_df.count()
deduplicated_df = combined_df.dropDuplicates(["shipment_id"])
after_dedup_count = deduplicated_df.count()

print("Rows before deduplication:", before_dedup_count)
print("Rows after deduplication:", after_dedup_count)
print("Rows removed:", before_dedup_count - after_dedup_count)

Repeated shipment IDs:
+-----------+-----+
|shipment_id|count|
+-----------+-----+
|SHP0000128 |2    |
|SHP0000254 |2    |
|SHP0000411 |2    |
|SHP0000455 |2    |
|SHP0000477 |2    |
|SHP0000550 |2    |
|SHP0000628 |2    |
|SHP0000774 |2    |
|SHP0000968 |2    |
|SHP0001041 |2    |
|SHP0001119 |2    |
|SHP0001166 |2    |
|SHP0001183 |2    |
|SHP0001222 |2    |
|SHP0001286 |2    |
|SHP0001448 |2    |
|SHP0001519 |2    |
|SHP0001535 |2    |
|SHP0001642 |2    |
|SHP0001737 |2    |
|SHP0002087 |2    |
|SHP0002213 |2    |
|SHP0002310 |2    |
|SHP0002494 |2    |
|SHP0002593 |2    |
|SHP0002680 |2    |
|SHP0002728 |2    |
|SHP0002777 |2    |
|SHP0002881 |2    |
|SHP0002969 |2    |
|SHP0003032 |2    |
|SHP0003048 |2    |
|SHP0003057 |2    |
|SHP0003174 |2    |
|SHP0003310 |2    |
|SHP0003355 |2    |
|SHP0003426 |2    |
|SHP0003496 |2    |
|SHP0003508 |2    |
|SHP0003621 |2    |
|SHP0003712 |2    |
|SHP0003800 |2    |
|SHP0003883 |2    |
|SHP0003887 |2    |
|SHP0004156 |2    |
|SHP0004294 |2   

In [40]:
sensor_exploded_df = (sensor_df.select("warehouse_id",
        F.col("environment.temperature_range_c.minimum").alias("min_temperature_c"),
        F.col("environment.temperature_range_c.maximum").alias("max_temperature_c"),
        F.explode("sensors").alias("sensor"))
    .select(
        "warehouse_id", "min_temperature_c", "max_temperature_c",
        F.col("sensor.sensor_id").alias("sensor_id"),
        F.col("sensor.type").alias("sensor_type"),
        F.col("sensor.enabled").alias("enabled")))
sensor_exploded_df.show(30, truncate=False)

+------------+-----------------+-----------------+---------+-----------+-------+
|warehouse_id|min_temperature_c|max_temperature_c|sensor_id|sensor_type|enabled|
+------------+-----------------+-----------------+---------+-----------+-------+
|W01         |18               |32               |W01-S1   |Temperature|false  |
|W01         |18               |32               |W01-S2   |Humidity   |true   |
|W01         |18               |32               |W01-S3   |Door       |true   |
|W02         |18               |32               |W02-S1   |Temperature|true   |
|W02         |18               |32               |W02-S2   |Humidity   |true   |
|W02         |18               |32               |W02-S3   |Door       |true   |
|W03         |18               |32               |W03-S1   |Temperature|true   |
|W03         |18               |32               |W03-S2   |Humidity   |true   |
|W03         |18               |32               |W03-S3   |Door       |true   |
|W04         |18            

**PART G | Lazy Execution, Caching, and Persistence**

In [41]:
ex21_df = (shipments_df
    .filter(F.col("shipment_status") == "Delivered")
    .join(warehouse_df.select("warehouse_id", "region"), "warehouse_id", "left")
    .withColumn(
        "fee_per_kg",
        F.when(
            F.col("package_weight_kg") > 0,
            F.col("shipping_fee_inr") / F.col("package_weight_kg")))
    .groupBy("region")
    .agg(
        F.count("*").alias("delivered_shipments"),
        F.avg("fee_per_kg").alias("avg_fee_per_kg"))
    .orderBy(F.desc("delivered_shipments")))
ex21_df.explain(mode="formatted")  # Inspect plan before triggering an action
ex21_df.show(truncate=False)

== Physical Plan ==
AdaptiveSparkPlan (14)
+- Sort (13)
   +- Exchange (12)
      +- HashAggregate (11)
         +- Exchange (10)
            +- HashAggregate (9)
               +- Project (8)
                  +- BroadcastHashJoin LeftOuter BuildRight (7)
                     :- Project (3)
                     :  +- Filter (2)
                     :     +- Scan csv  (1)
                     +- BroadcastExchange (6)
                        +- Filter (5)
                           +- Scan csv  (4)


(1) Scan csv 
Output [4]: [warehouse_id#22, package_weight_kg#27, shipping_fee_inr#28, shipment_status#30]
Batched: false
Location: InMemoryFileIndex [file:/content/logistics_data/shipments_75000.csv]
PushedFilters: [IsNotNull(shipment_status), EqualTo(shipment_status,Delivered)]
ReadSchema: struct<warehouse_id:string,package_weight_kg:double,shipping_fee_inr:double,shipment_status:string>

(2) Filter
Input [4]: [warehouse_id#22, package_weight_kg#27, shipping_fee_inr#28, shipment_status#30

In [42]:
delivered_cached_df = shipments_df.filter(
    F.col("shipment_status") == "Delivered").cache()

start = time.time()
delivered_count_1 = delivered_cached_df.count()
time_count_1 = time.time() - start

start = time.time()
delivered_count_2 = delivered_cached_df.count()
time_count_2 = time.time() - start

start = time.time()
delivered_fees = delivered_cached_df.agg(
    F.sum("shipping_fee_inr").alias("total_fees")).collect()
time_agg = time.time() - start

print(f"Count 1: {delivered_count_1:,} rows in {time_count_1:.3f}s")
print(f"Count 2: {delivered_count_2:,} rows in {time_count_2:.3f}s")
print(f"Aggregation result: {delivered_fees}; time: {time_agg:.3f}s")

delivered_cached_df.unpersist()

Count 1: 52,434 rows in 2.312s
Count 2: 52,434 rows in 0.080s
Aggregation result: [Row(total_fees=19528933.499999907)]; time: 0.419s


DataFrame[shipment_id: string, booking_date: date, dispatch_date: date, promised_delivery_date: date, actual_delivery_date: date, warehouse_id: string, carrier_id: string, route_id: string, customer_segment: string, service_level: string, package_weight_kg: double, shipping_fee_inr: double, shipping_cost_inr: double, shipment_status: string, delivery_attempts: int, damage_flag: string, customer_rating: double, handling_minutes: double, payment_mode: string, priority_flag: string]

In [43]:
express_persisted_df = shipments_df.filter(F.col("service_level") == "Express").persist(StorageLevel.MEMORY_AND_DISK)

print("Express shipment count:", express_persisted_df.count())

express_persisted_df.groupBy("shipment_status").count().show()

print("Storage level:", express_persisted_df.storageLevel)
express_persisted_df.unpersist()
print("Express shipment DataFrame unpersisted.")

Express shipment count: 21090
+---------------+-----+
|shipment_status|count|
+---------------+-----+
|       Returned| 1767|
|           Lost|  575|
|     In Transit| 2944|
|      Cancelled|  985|
|      Delivered|14819|
+---------------+-----+

Storage level: Disk Memory Serialized 1x Replicated
Express shipment DataFrame unpersisted.


**PART H | Partitioning and Optimization**

In [44]:
shipments_8_df = shipments_df.repartition(8)
print("Partition count:", shipments_8_df.rdd.getNumPartitions())

ex24_df = (shipments_8_df
    .withColumn("partition_id", F.spark_partition_id())
    .groupBy("partition_id")
    .count()
    .orderBy("partition_id"))
ex24_df.show()

Partition count: 8
+------------+-----+
|partition_id|count|
+------------+-----+
|           0| 9375|
|           1| 9374|
|           2| 9376|
|           3| 9375|
|           4| 9375|
|           5| 9375|
|           6| 9375|
|           7| 9375|
+------------+-----+



In [45]:
shipments_12_df = shipments_df.repartition(12)
shipments_coalesce_3_df = shipments_12_df.coalesce(3)
shipments_repartition_3_df = shipments_12_df.repartition(3)

print("Initial partitions:", shipments_12_df.rdd.getNumPartitions())
print("After coalesce(3):", shipments_coalesce_3_df.rdd.getNumPartitions())
print("After repartition(3):", shipments_repartition_3_df.rdd.getNumPartitions())

print("\n--- coalesce(3) plan ---")
shipments_coalesce_3_df.explain(mode="formatted")

print("\n--- repartition(3) plan ---")
shipments_repartition_3_df.explain(mode="formatted")

Initial partitions: 12
After coalesce(3): 3
After repartition(3): 3

--- coalesce(3) plan ---
== Physical Plan ==
AdaptiveSparkPlan (7)
+- == Final Plan ==
   ResultQueryStage (5)
   +- Coalesce (4)
      +- ShuffleQueryStage (3), Statistics(sizeInBytes=19.1 MiB, rowCount=7.50E+4)
         +- Exchange (2)
            +- Scan csv  (1)
+- == Initial Plan ==
   Coalesce (6)
   +- Exchange (2)
      +- Scan csv  (1)


(1) Scan csv 
Output [20]: [shipment_id#17, booking_date#18, dispatch_date#19, promised_delivery_date#20, actual_delivery_date#21, warehouse_id#22, carrier_id#23, route_id#24, customer_segment#25, service_level#26, package_weight_kg#27, shipping_fee_inr#28, shipping_cost_inr#29, shipment_status#30, delivery_attempts#31, damage_flag#32, customer_rating#33, handling_minutes#34, payment_mode#35, priority_flag#36]
Batched: false
Location: InMemoryFileIndex [file:/content/logistics_data/shipments_75000.csv]
ReadSchema: struct<shipment_id:string,booking_date:date,dispatch_date:date

In [46]:
shipment_warehouse_df = shipments_df.join(warehouse_df, on="warehouse_id", how="left")

parquet_path = "/content/logistics_shipments_warehouse_parquet"
(shipment_warehouse_df
    .repartition(4)
    .write
    .mode("overwrite")
    .parquet(parquet_path))

print("Parquet output path:", parquet_path)
print("Generated files:")
for name in sorted(os.listdir(parquet_path)):
    print(name)

Parquet output path: /content/logistics_shipments_warehouse_parquet
Generated files:
._SUCCESS.crc
.part-00000-65f02aa4-5dc8-4e03-af8e-46a111791f84-c000.snappy.parquet.crc
.part-00001-65f02aa4-5dc8-4e03-af8e-46a111791f84-c000.snappy.parquet.crc
.part-00002-65f02aa4-5dc8-4e03-af8e-46a111791f84-c000.snappy.parquet.crc
.part-00003-65f02aa4-5dc8-4e03-af8e-46a111791f84-c000.snappy.parquet.crc
_SUCCESS
part-00000-65f02aa4-5dc8-4e03-af8e-46a111791f84-c000.snappy.parquet
part-00001-65f02aa4-5dc8-4e03-af8e-46a111791f84-c000.snappy.parquet
part-00002-65f02aa4-5dc8-4e03-af8e-46a111791f84-c000.snappy.parquet
part-00003-65f02aa4-5dc8-4e03-af8e-46a111791f84-c000.snappy.parquet


**FINAL CHALLENGE | Logistics Performance Pipeline**

In [48]:

pipeline_df = (shipments_df
    .join(warehouse_df, on="warehouse_id", how="left")
    .join(carrier_df, on="carrier_id", how="left")
    .join(route_df, on="route_id", how="left"))

print("Joined DataFrame schema:")
pipeline_df.printSchema()

print("Sample joined records:")
pipeline_df.show(10, truncate=False)

Joined DataFrame schema:
root
 |-- route_id: string (nullable = true)
 |-- carrier_id: string (nullable = true)
 |-- warehouse_id: string (nullable = true)
 |-- shipment_id: string (nullable = true)
 |-- booking_date: date (nullable = true)
 |-- dispatch_date: date (nullable = true)
 |-- promised_delivery_date: date (nullable = true)
 |-- actual_delivery_date: date (nullable = true)
 |-- customer_segment: string (nullable = true)
 |-- service_level: string (nullable = true)
 |-- package_weight_kg: double (nullable = true)
 |-- shipping_fee_inr: double (nullable = true)
 |-- shipping_cost_inr: double (nullable = true)
 |-- shipment_status: string (nullable = true)
 |-- delivery_attempts: integer (nullable = true)
 |-- damage_flag: string (nullable = true)
 |-- customer_rating: double (nullable = true)
 |-- handling_minutes: double (nullable = true)
 |-- payment_mode: string (nullable = true)
 |-- priority_flag: string (nullable = true)
 |-- warehouse_name: string (nullable = true)
 |-- 

In [49]:
median_cost = shipments_df.approxQuantile("shipping_cost_inr", [0.5], 0.01)[0]

median_weight = shipments_df.approxQuantile(
    "package_weight_kg", [0.5], 0.01)[0]

pipeline_clean_df = (pipeline_df.withColumn(
        "shipping_cost_clean_inr",
        F.when(
            F.col("shipping_cost_inr").isNotNull(),
            F.col("shipping_cost_inr"))
        .when(
            F.col("base_cost_per_km").isNotNull()
            & F.col("distance_km").isNotNull(),
            F.col("base_cost_per_km") * F.col("distance_km"))
        .otherwise(F.lit(float(median_cost))))
    .withColumn(
        "package_weight_clean_kg",
        F.coalesce(
            F.col("package_weight_kg"),
            F.lit(float(median_weight)))))

print("Median shipping cost:", median_cost)
print("Median package weight:", median_weight)

pipeline_clean_df.select(
    "shipment_id",
    "shipping_cost_inr",
    "shipping_cost_clean_inr",
    "package_weight_kg",
    "package_weight_clean_kg").show(10, truncate=False)

Median shipping cost: 267.9
Median package weight: 2.83
+-----------+-----------------+-----------------------+-----------------+-----------------------+
|shipment_id|shipping_cost_inr|shipping_cost_clean_inr|package_weight_kg|package_weight_clean_kg|
+-----------+-----------------+-----------------------+-----------------+-----------------------+
|SHP0000001 |324.9            |324.9                  |3.93             |3.93                   |
|SHP0000002 |351.6            |351.6                  |5.62             |5.62                   |
|SHP0000003 |521.4            |521.4                  |8.78             |8.78                   |
|SHP0000004 |160.1            |160.1                  |0.97             |0.97                   |
|SHP0000005 |265.8            |265.8                  |NULL             |2.83                   |
|SHP0000006 |499.5            |499.5                  |6.05             |6.05                   |
|SHP0000007 |205.8            |205.8                  |1.36   

In [50]:
pipeline_metrics_df = pipeline_clean_df.withColumn(
    "delivery_duration_days",
    F.when(
        F.col("dispatch_date").isNotNull()
        & F.col("actual_delivery_date").isNotNull(),
        F.datediff(
            F.col("actual_delivery_date"),
            F.col("dispatch_date"))))

pipeline_metrics_df.select(
    "shipment_id",
    "dispatch_date",
    "actual_delivery_date",
    "delivery_duration_days").show(15, truncate=False)

+-----------+-------------+--------------------+----------------------+
|shipment_id|dispatch_date|actual_delivery_date|delivery_duration_days|
+-----------+-------------+--------------------+----------------------+
|SHP0000001 |2026-05-24   |2026-05-27          |3                     |
|SHP0000002 |2026-05-18   |2026-05-21          |3                     |
|SHP0000003 |2026-02-01   |2026-02-04          |3                     |
|SHP0000004 |2026-03-12   |2026-03-18          |6                     |
|SHP0000005 |2026-08-16   |2026-08-29          |13                    |
|SHP0000006 |2026-02-08   |2026-02-09          |1                     |
|SHP0000007 |2026-01-11   |NULL                |NULL                  |
|SHP0000008 |2026-03-06   |NULL                |NULL                  |
|SHP0000009 |2026-08-12   |2026-08-21          |9                     |
|SHP0000010 |2026-06-02   |2026-06-11          |9                     |
|SHP0000011 |2026-04-27   |2026-05-01          |4               

In [52]:
pipeline_metrics_df = (
    pipeline_metrics_df
    .withColumn(
        "late_delivery_indicator",
        F.when(
            F.col("actual_delivery_date").isNull()
            | F.col("promised_delivery_date").isNull(),
            F.lit(None).cast("int"))
        .when(
            F.col("actual_delivery_date")
            > F.col("promised_delivery_date"),1).otherwise(0))
    .withColumn(
        "on_time_delivery_indicator",
        F.when(
            F.col("actual_delivery_date").isNull()
            | F.col("promised_delivery_date").isNull(),
            F.lit(None).cast("int"))
        .when(
            F.col("actual_delivery_date")
            <= F.col("promised_delivery_date"),1).otherwise(0)))

pipeline_metrics_df.select(
    "shipment_id",
    "promised_delivery_date",
    "actual_delivery_date",
    "late_delivery_indicator",
    "on_time_delivery_indicator").show(15, truncate=False)

+-----------+----------------------+--------------------+-----------------------+--------------------------+
|shipment_id|promised_delivery_date|actual_delivery_date|late_delivery_indicator|on_time_delivery_indicator|
+-----------+----------------------+--------------------+-----------------------+--------------------------+
|SHP0000001 |2026-05-26            |2026-05-27          |1                      |0                         |
|SHP0000002 |2026-05-21            |2026-05-21          |0                      |1                         |
|SHP0000003 |2026-02-03            |2026-02-04          |1                      |0                         |
|SHP0000004 |2026-03-17            |2026-03-18          |1                      |0                         |
|SHP0000005 |2026-08-25            |2026-08-29          |1                      |0                         |
|SHP0000006 |2026-02-09            |2026-02-09          |0                      |1                         |
|SHP0000007 |2026-0

In [53]:
pipeline_metrics_df = pipeline_metrics_df.withColumn(
    "shipment_profit_inr",
    F.col("shipping_fee_inr")
    - F.col("shipping_cost_clean_inr"))

pipeline_metrics_df.select(
    "shipment_id",
    "shipping_fee_inr",
    "shipping_cost_clean_inr",
    "shipment_profit_inr").show(15, truncate=False)

+-----------+----------------+-----------------------+-------------------+
|shipment_id|shipping_fee_inr|shipping_cost_clean_inr|shipment_profit_inr|
+-----------+----------------+-----------------------+-------------------+
|SHP0000001 |554.34          |324.9                  |229.44000000000005 |
|SHP0000002 |414.56          |351.6                  |62.95999999999998  |
|SHP0000003 |724.64          |521.4                  |203.24             |
|SHP0000004 |287.86          |160.1                  |127.76000000000002 |
|SHP0000005 |251.68          |265.8                  |-14.120000000000005|
|SHP0000006 |477.9           |499.5                  |-21.600000000000023|
|SHP0000007 |298.68          |205.8                  |92.88              |
|SHP0000008 |284.52          |210.2                  |74.32              |
|SHP0000009 |861.74          |712.9                  |148.84000000000003 |
|SHP0000010 |309.76          |149.6                  |160.16             |
|SHP0000011 |427.06      

In [54]:
final_agg_df = (
    pipeline_metrics_df
    .groupBy(
        "region",
        "carrier_id",
        "carrier_name").agg(
        F.sum("shipping_fee_inr").alias("total_revenue_inr"),

        F.sum("shipment_profit_inr").alias("total_profit_inr"),

        F.count("*").alias("shipment_count"),F.avg(
            "late_delivery_indicator").alias("late_delivery_rate"),

        F.avg(
            "on_time_delivery_indicator").alias("on_time_delivery_rate"))
    .withColumn(
        "late_delivery_percentage",
        F.col("late_delivery_rate") * 100)
    .withColumn(
        "on_time_delivery_percentage",
        F.col("on_time_delivery_rate") * 100).drop("late_delivery_rate","on_time_delivery_rate" ))

final_agg_df.show(truncate=False)

+------+----------+--------------------+------------------+-------------------+--------------+------------------------+---------------------------+
|region|carrier_id|carrier_name        |total_revenue_inr |total_profit_inr   |shipment_count|late_delivery_percentage|on_time_delivery_percentage|
+------+----------+--------------------+------------------+-------------------+--------------+------------------------+---------------------------+
|North |C07       |BluePeak Delivery   |425396.69999999995|40825.79999999997  |1140          |39.0402075226978        |60.95979247730221          |
|West  |C05       |CargoNest           |1150047.2599999993|38854.860000000015 |3092          |41.863699582753824      |58.136300417246176         |
|South |C04       |AirSprint Couriers  |1366979.28        |-263752.6199999997 |3602          |40.57443365695793       |59.42556634304207          |
|East  |C04       |AirSprint Couriers  |211583.64000000004|-47550.259999999966|561           |40.64171122994652 

In [55]:
region_window = (
    Window
    .partitionBy("region")
    .orderBy(
        F.col("on_time_delivery_percentage").desc_nulls_last()))

final_report_df = final_agg_df.withColumn(
    "carrier_rank",
    F.dense_rank().over(region_window))

final_report_df.orderBy(
    "region",
    "carrier_rank").show(truncate=False)

+------+----------+--------------------+------------------+-------------------+--------------+------------------------+---------------------------+------------+
|region|carrier_id|carrier_name        |total_revenue_inr |total_profit_inr   |shipment_count|late_delivery_percentage|on_time_delivery_percentage|carrier_rank|
+------+----------+--------------------+------------------+-------------------+--------------+------------------------+---------------------------+------------+
|East  |C01       |SwiftLine Express   |393862.61999999994|-15339.379999999986|1037          |38.40877914951989       |61.59122085048011          |1           |
|East  |C04       |AirSprint Couriers  |211583.64000000004|-47550.259999999966|561           |40.64171122994652       |59.35828877005348          |2           |
|East  |C02       |ParcelBridge        |373590.02         |9262.12000000001   |1022          |41.9170243204578        |58.0829756795422           |3           |
|East  |C05       |CargoNest      

In [56]:
final_report_cached_df = final_report_df.cache()

total_rows = final_report_cached_df.count()

print("Total rows in final report:", total_rows)
print("Storage level:", final_report_cached_df.storageLevel)

Total rows in final report: 32
Storage level: Disk Memory Deserialized 1x Replicated


In [57]:
print("Final logistics performance report:")

final_report_cached_df.orderBy(
    "region",
    "carrier_rank").show(truncate=False)

print("Total revenue and profit by region:")

final_report_cached_df.groupBy(
    "region").agg(
    F.sum("total_revenue_inr").alias("regional_revenue_inr"),
    F.sum("total_profit_inr").alias("regional_profit_inr")).orderBy("region").show(truncate=False)

Final logistics performance report:
+------+----------+--------------------+------------------+-------------------+--------------+------------------------+---------------------------+------------+
|region|carrier_id|carrier_name        |total_revenue_inr |total_profit_inr   |shipment_count|late_delivery_percentage|on_time_delivery_percentage|carrier_rank|
+------+----------+--------------------+------------------+-------------------+--------------+------------------------+---------------------------+------------+
|East  |C01       |SwiftLine Express   |393862.61999999994|-15339.379999999986|1037          |38.40877914951989       |61.59122085048011          |1           |
|East  |C04       |AirSprint Couriers  |211583.64000000004|-47550.259999999966|561           |40.64171122994652       |59.35828877005348          |2           |
|East  |C02       |ParcelBridge        |373590.02         |9262.12000000001   |1022          |41.9170243204578        |58.0829756795422           |3           

In [58]:
output_path = "/content/logistics_performance_report"
(   final_report_cached_df
    .write
    .mode("overwrite")
    .partitionBy("region")
    .parquet(output_path))

print("Final report saved successfully!")
print("Output location:", output_path)

Final report saved successfully!
Output location: /content/logistics_performance_report


In [59]:
import os

for root, dirs, files_list in os.walk(output_path):
    print(root)

    for file in files_list:
        print("   ", file)

verified_df = spark.read.parquet(output_path)

print("Rows read from Parquet:", verified_df.count())

verified_df.orderBy(
    "region",
    "carrier_rank").show(truncate=False)

/content/logistics_performance_report
    ._SUCCESS.crc
    _SUCCESS
/content/logistics_performance_report/region=West
    part-00164-3932ccd6-e8d9-4d12-9d9e-a6b35c030d7d.c000.snappy.parquet
    .part-00164-3932ccd6-e8d9-4d12-9d9e-a6b35c030d7d.c000.snappy.parquet.crc
/content/logistics_performance_report/region=North
    .part-00192-3932ccd6-e8d9-4d12-9d9e-a6b35c030d7d.c000.snappy.parquet.crc
    part-00192-3932ccd6-e8d9-4d12-9d9e-a6b35c030d7d.c000.snappy.parquet
/content/logistics_performance_report/region=South
    part-00090-3932ccd6-e8d9-4d12-9d9e-a6b35c030d7d.c000.snappy.parquet
    .part-00090-3932ccd6-e8d9-4d12-9d9e-a6b35c030d7d.c000.snappy.parquet.crc
/content/logistics_performance_report/region=East
    .part-00129-3932ccd6-e8d9-4d12-9d9e-a6b35c030d7d.c000.snappy.parquet.crc
    part-00129-3932ccd6-e8d9-4d12-9d9e-a6b35c030d7d.c000.snappy.parquet
Rows read from Parquet: 32
+----------+--------------------+------------------+-------------------+--------------+-------------------

In [60]:
final_report_cached_df.unpersist()

print("Final report cache released successfully.")

Final report cache released successfully.
